# 🛒 Análisis ECLAT — Hábitos de Compra

**Presentación ejecutiva y técnica** de minería de datos para hábitos de compra de hogares colombianos.

Este notebook integra buenas prácticas extraídas de la implementación por fases en `scripts/eclat.ipynb` y las adapta a un flujo robusto para datos reales (texto sucio, normalización lingüística y segmentación demográfica).

## Agenda de presentación

1. Setup y parámetros de negocio
2. Carga y calidad de datos
3. Normalización de texto de productos
4. Matriz transaccional (one-hot)
5. ECLAT (núcleo del modelo)
6. Reglas de asociación y priorización
7. Visualizaciones para storytelling
8. Segmentación (estrato y grupo etario)
9. Hallazgos accionables y conclusiones

---

## 0. Instalación y Setup

In [ ]:
# Propósito: instalar dependencias solo si el entorno no las tiene.
# !pip install pandas openpyxl spacy rapidfuzz matplotlib seaborn networkx scikit-learn
# !python -m spacy download es_core_news_sm

In [ ]:
from __future__ import annotations

import re
import unicodedata
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns
import spacy
from sklearn.preprocessing import MultiLabelBinarizer

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

# ── Configuración global ──────────────────────────────
DATASET_PATH = Path('data/dataset.xlsx')
SHEET_NAME = 'Respuestas de formulario 1'
MIN_SUPPORT = 0.2
MAX_ITEMSET = 3
EDAD_BINS = [0, 25, 35, 50, 100]
EDAD_LABELS = ['Joven', 'Adulto joven', 'Adulto', 'Senior']
OUTPUT_PATH = Path('df_procesado.xlsx')
# ─────────────────────────────────────────────────────

try:
    nlp = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
except OSError as exc:
    raise RuntimeError(
        'No se encontró el modelo es_core_news_sm. Ejecuta: python -m spacy download es_core_news_sm'
    ) from exc

## 1. Carga de Datos

**Fase equivalente en `scripts/eclat.ipynb`:** lectura base del dataset y separación demográfica vs productos.

En esta versión se añade:
- renombrado robusto de columnas,
- tipado de variables,
- creación de `grupo_edad`,
- validaciones tempranas para evitar errores de mezcla de tipos en fases posteriores.

In [ ]:
dataset_file = DATASET_PATH
if not dataset_file.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en {DATASET_PATH}"
    )

df = pd.read_excel(dataset_file, sheet_name=SHEET_NAME)
df.head()

In [ ]:
# Propósito: cargar el dataset, renombrar columnas y crear grupo de edad.
column_rename_map = {
    "marca temporal": "fecha",
    "edad": "edad",
    "genero": "genero",
    "estrato nse": "estrato_nse",
    "ciudad de origen": "ciudad_origen",
    "con quien vive actualmente": "quien_vive",
    "numero de personas en el hogar": "num_personas",
    "quien realiza las compras en el hogar": "quien_compra",
    "tu misión de compra...": "mision_compra",
    "lugar principal de compra": "canal_compra",
    "presupuesto mensual aproximado": "gasto",
    "alimentos básicos": "alimentos_basicos",
    "proteinas": "proteinas",
    "bebidas": "bebidas",
    "snacks y otros": "snacks",
    "delikatessen": "delikatessen",
    "aseo para el hogar": "aseo_hogar",
    "aseo personal": "aseo_personal",
    "ultima": "otros",
}

rows_to_cols = [
    "alimentos_basicos",
    "proteinas",
    "bebidas",
    "snacks",
    "delikatessen",
    "aseo_hogar",
    "aseo_personal",
    "otros",
]

cols_demograficas = [
    "fecha",
    "edad",
    "genero",
    "estrato_nse",
    "ciudad_origen",
    "quien_vive",
    "num_personas",
    "quien_compra",
    "mision_compra",
    "canal_compra",
    "gasto",
    "grupo_edad",
]

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns=column_rename_map)

# Renombrado robusto para variantes largas de la columna de misión de compra.
if "mision_compra" not in df.columns:
    mission_candidates = [
        col
        for col in df.columns
        if col.startswith("tu misión de compra")
        or col.startswith("tu mision de compra")
    ]
    if mission_candidates:
        df = df.rename(columns={mission_candidates[0]: "mision_compra"})

df.head()

In [ ]:
df["edad"] = pd.to_numeric(df["edad"], errors="coerce")
df["gasto"] = pd.to_numeric(df["gasto"], errors="coerce")
df["grupo_edad"] = pd.cut(
    df["edad"], bins=EDAD_BINS, labels=EDAD_LABELS, include_lowest=True
)

print("Shape:", df.shape)
df.info()
df.head(3)

## 2. Normalización de Texto

In [ ]:
# Propósito: definir pipeline de limpieza y normalización de productos en tres capas.
replace_map: dict[str, str | None] = {
    "arroz diana": "arroz",
    "suavitel": "suavizante",
    "1200k": None,
    "700k": None,
    "bananos": "banano",
    "frijoles": "frijol",
    "jabon": "jabon tocador",
}

In [29]:
comparison_rows = []

def normalize_product(token: str) -> str | None:
    """Normaliza un token de producto con lematización + diccionario replace_map."""
    if not token or token == "ninguno":
        return None

    doc = nlp(token)
    lemma = doc[0].lemma_ if doc else token
    lemma = lemma.strip()

    result = replace_map.get(lemma, lemma)
    return result if result else None


def clean_and_split(text: str) -> list[str]:
    """Limpia texto crudo, separa productos y devuelve lista única normalizada.
       Además registra trazabilidad original vs normalizado."""
    
    if pd.isna(text) or not isinstance(text, str):
        return []

    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("utf-8", "ignore")
    text = text.lower()
    text = re.sub(r"[\n\r]+", ",", text)
    text = re.sub(r"-\s*", ",", text)
    text = re.sub(r";", ",", text)
    text = re.sub(r"\s{2,}", ",", text)
    text = re.sub(r",+", ",", text)

    tokens = [t.strip() for t in text.split(",") if t.strip()]
    tokens = [t for t in tokens if t != "ninguno"]

    normalized = []

    for t in tokens:
        norm = normalize_product(t)
        
        # Guardar trazabilidad (aunque sea None)
        comparison_rows.append({
            "original": t,
            "normalized": norm
        })
        
        if norm:
            normalized.append(norm)

    return sorted(set(normalized))

In [30]:
# Propósito: transformar columnas de productos en matriz one-hot y consolidar base final.
processed = df[rows_to_cols].apply(lambda col: col.map(clean_and_split))
processed.head()

,alimentos_basicos,proteinas,bebidas,snacks,delikatessen,aseo_hogar,aseo_personal,otros
0,"[lenteja, pasta]",[atun],"[leche, tú]",[],[],"[blanqueador, papel, suavizant]","[colonia, loción, talco]","[atun, blanqueador, colonia, desinfectante, la..."
1,"[frijol, pasta]","[carne, pescado, pollo]",[cafar],[],"[jamon, salsa]","[blanqueador, limpiapiso, limpiavidrio]","[acondicionador, jabon tocador, shampoo, talco]","[acondicionador, carne, frijol, fruta, jabon t..."
2,[pasta],"[carne, huevo]","[gaseós, leche]",[],[otro],"[cera, papel, servilleta, suavizant]","[acondicionador, crema, jabon tocador]","[arroz, atun, blanqueador, cera, crema, deterg..."
3,"[frijol, harina, lenteja, pasta]","[carne, pollo]","[cafar, leche]",[],[salsa],"[blanqueador, cera, desinfectante, detergente,...","[colonia, hilo, jabon tocador, loción, perfumi...","[arroz, carne, desinfectante, detergente, frij..."
4,[lenteja],"[huevo, pollo]",[agua],[],"[cerveza, queso, salsa]","[ambientador, blanqueador, cera, desinfectante...","[crema, enjuagar, jabon tocador, proteccion, s...","[aceite, blanqueador, cerveza, desinfectante, ..."


In [38]:
df_comparison = pd.DataFrame(comparison_rows)
df_comparison = df_comparison.drop_duplicates()

display(df_comparison)

,original,normalized
0,pastas,pasta
1,lentejas,lenteja
2,frijoles,frijol
7,harinas,harina
11,arroz,arroz
...,...,...
3615,azucar,azucar
3632,suavitel,suavizante
3713,harina,harina
3771,sal,sal


In [39]:
combined = processed.apply(lambda x: sum(x.tolist(), []), axis=1)
print(combined)

0      [lenteja, pasta, atun, leche, tú, blanqueador,...
1      [frijol, pasta, carne, pescado, pollo, cafar, ...
2      [pasta, carne, huevo, gaseós, leche, otro, cer...
3      [frijol, harina, lenteja, pasta, carne, pollo,...
4      [lenteja, huevo, pollo, agua, cerveza, queso, ...
                             ...                        
195    [arroz, pescado, cafar, leche, galleta, panela...
196    [frijol, garbanzo, lenteja, atun, pollo, agua,...
197    [lenteja, pasta, carne, pollo, agua, gaseós, c...
198    [arroz, frijol, garbanzo, harina, pasta, carne...
199    [arroz, frijol, harina, lenteja, pasta, carne,...
Length: 200, dtype: object


## 3. Codificación One-Hot

In [40]:
mlb = MultiLabelBinarizer()
one_hot_array = mlb.fit_transform(combined)
df_one_hot = pd.DataFrame(one_hot_array, columns=mlb.classes_, index=df.index)
PRODUCT_COLUMNS = list(df_one_hot.columns)

df_one_hot.head()

,aceite,acondicionador,agua,ambientador,arroz,atun,azucar,blanqueador,cafar,carne,cera,cereal,cerveza,chocolate,colonia,crema,desinfectante,desodorant,detergente,dulce,enjuagar,frijol,fruta,galleta,garbanzo,gaseós,harina,helado,hilo,huevo,jabon tocador,jamon,jugo,lavalozar,leche,lenteja,limpiapiso,limpiavidrio,loción,otro,pan,panelar,papas,papel,pasta,perfumir,pescado,pollo,poncar,proteccion,queso,sal,salchicha,salsa,servilleta,shampoo,suavizant,suavizante,talco,tú,verdura,vino
0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0
1,0,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0
2,0,1,0,0,1,1,0,1,0,1,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0
3,0,0,0,0,1,0,0,1,1,1,1,0,0,0,1,0,1,0,1,0,0,1,0,0,0,0,1,0,1,0,1,0,0,0,1,1,0,0,1,0,1,0,0,0,1,1,0,1,0,1,0,0,0,1,1,0,0,0,0,0,0,0
4,1,0,1,1,0,0,0,1,0,0,1,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,1,0,1,1,0,0,0,0,0


In [41]:
df_base = df.drop(columns=rows_to_cols)
df_final = pd.concat([df_base, df_one_hot], axis=1)
df_final.head()

,fecha,edad,genero,estrato_nse,ciudad_origen,quien_vive,num_personas,quien_compra,mision_compra,canal_compra,gasto,grupo_edad,aceite,acondicionador,agua,ambientador,arroz,atun,azucar,blanqueador,cafar,carne,cera,cereal,cerveza,chocolate,colonia,crema,desinfectante,desodorant,detergente,dulce,enjuagar,frijol,fruta,galleta,garbanzo,gaseós,harina,helado,hilo,huevo,jabon tocador,jamon,jugo,lavalozar,leche,lenteja,limpiapiso,limpiavidrio,loción,otro,pan,panelar,papas,papel,pasta,perfumir,pescado,pollo,poncar,proteccion,queso,sal,salchicha,salsa,servilleta,shampoo,suavizant,suavizante,talco,tú,verdura,vino
0,2026-04-01 08:00:00,18,Masculino,E2,Bogotá,Padres,5,Otra persona,Quincenal,"Supermercado (Exito, Jumbo, Etc)",700000.0,Joven,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0
1,2026-04-01 13:42:00,21,Femenino,E3,Medellín,Padres,4,Otra persona,Mensual,"Hard Discount (D1, ARA, etc )",1000000.0,Joven,0,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0
2,2026-04-01 14:54:00,20,Masculino,E3,Santa Marta,Padres,2,Compartido,Quincenal,Tienda de Barrio,500000.0,Joven,0,1,0,0,1,1,0,1,0,1,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0
3,2026-04-01 15:21:00,25,Masculino,E3,Medellín,Familia,6,Compartido,Mensual,"Supermercado (Exito, Jumbo, Etc)",2000000.0,Joven,0,0,0,0,1,0,0,1,1,1,1,0,0,0,1,0,1,0,1,0,0,1,0,0,0,0,1,0,1,0,1,0,0,0,1,1,0,0,1,0,1,0,0,0,1,1,0,1,0,1,0,0,0,1,1,0,0,0,0,0,0,0
4,2026-04-01 21:55:00,21,Masculino,E3,Bogotá,Familia,4,Yo,Mensual,"Supermercado (Exito, Jumbo, Etc)",1200000.0,Joven,1,0,1,1,0,0,0,1,0,0,1,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,1,0,1,1,0,0,0,0,0


In [44]:
df_final.to_excel(OUTPUT_PATH, index=False)

print("Filas x columnas:", df_one_hot.shape)
print("Filas x columnas final:", df_final.shape)
print("Total productos:", len(df_one_hot.columns))
display(df_final.head(3))

Filas x columnas: (200, 62)
Filas x columnas final: (200, 74)
Total productos: 62


,fecha,edad,genero,estrato_nse,ciudad_origen,quien_vive,num_personas,quien_compra,mision_compra,canal_compra,gasto,grupo_edad,aceite,acondicionador,agua,ambientador,arroz,atun,azucar,blanqueador,cafar,carne,cera,cereal,cerveza,chocolate,colonia,crema,desinfectante,desodorant,detergente,dulce,enjuagar,frijol,fruta,galleta,garbanzo,gaseós,harina,helado,hilo,huevo,jabon tocador,jamon,jugo,lavalozar,leche,lenteja,limpiapiso,limpiavidrio,loción,otro,pan,panelar,papas,papel,pasta,perfumir,pescado,pollo,poncar,proteccion,queso,sal,salchicha,salsa,servilleta,shampoo,suavizant,suavizante,talco,tú,verdura,vino
0,2026-04-01 08:00:00,18,Masculino,E2,Bogotá,Padres,5,Otra persona,Quincenal,"Supermercado (Exito, Jumbo, Etc)",700000.0,Joven,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0
1,2026-04-01 13:42:00,21,Femenino,E3,Medellín,Padres,4,Otra persona,Mensual,"Hard Discount (D1, ARA, etc )",1000000.0,Joven,0,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0
2,2026-04-01 14:54:00,20,Masculino,E3,Santa Marta,Padres,2,Compartido,Quincenal,Tienda de Barrio,500000.0,Joven,0,1,0,0,1,1,0,1,0,1,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0


## 4. Análisis Exploratorio (frecuencias)

Aquí usamos la matriz one-hot como única fuente de productos para garantizar consistencia analítica.

In [ ]:
# Propósito: identificar productos más frecuentes y su soporte para contexto exploratorio inicial.
cols_productos = PRODUCT_COLUMNS

productos_df = df_final[cols_productos]
freq_productos = productos_df.sum(numeric_only=True)

freq_productos_df = pd.DataFrame(
    
    {"producto": cols_productos, "frecuencia": freq_productos}
).sort_values(by="frecuencia", ascending=False)

freq_productos_df.head(20)

,producto,frecuencia
carne,carne,148
crema,crema,127
frijol,frijol,113
arroz,arroz,113
pollo,pollo,110
pasta,pasta,107
harina,harina,106
lenteja,lenteja,105
leche,leche,102
cafar,cafar,101


In [53]:
support_productos = (freq_productos / len(productos_df)).round(4)

resumen_frecuencia = pd.DataFrame(
    {
        'frecuencia': freq_productos,
        'support': support_productos,
    }
).sort_values('support', ascending=False)

print('Productos detectados:', len(cols_productos))
print('Transacciones:', len(productos_df))
display(resumen_frecuencia.head(20))

Productos detectados: 62
Transacciones: 200


,frecuencia,support
carne,148,0.740
crema,127,0.635
frijol,113,0.565
arroz,113,0.565
pollo,110,0.550
pasta,107,0.535
harina,106,0.530
lenteja,105,0.525
leche,102,0.510
cafar,101,0.505


## 5. ECLAT — Dataset Completo

**Fase equivalente en `scripts/eclat.ipynb`:** construcción de transacciones, TID lists e itemsets frecuentes.

Mejoras aplicadas en esta implementación:
- columnas de producto explícitas (`PRODUCT_COLUMNS`),
- intersección con *early-exit* cuando no quedan TIDs,
- `support_cache` con `frozenset` para consultas O(1),
- salida ordenada y reproducible para exposición.

### 5.1 Formalización del algoritmo ECLAT

Esta celda define las funciones nucleares del modelo. Primero se transforma la matriz binaria en transacciones; luego se ejecuta ECLAT en formato vertical (TID-lists) para estimar soporte de itemsets frecuentes de manera eficiente y reproducible.

In [55]:
# Propósito: implementar ECLAT reusable con estructura vertical y cache de soporte.
def build_transactions(product_binary_df: pd.DataFrame) -> list[list[str]]:
    """Convierte matriz binaria de productos a lista de transacciones."""
    return [
        list(product_binary_df.columns[row.values == 1])
        for _, row in product_binary_df.iterrows()
    ]

def run_eclat(
    data: pd.DataFrame,
    min_support: float = MIN_SUPPORT,
    max_len: int = MAX_ITEMSET,
) -> tuple[pd.DataFrame, dict[frozenset[str], float], pd.DataFrame]:
    """Ejecuta ECLAT y retorna itemsets frecuentes, cache de soporte y dataframe de productos."""
    product_cols = [c for c in PRODUCT_COLUMNS if c in data.columns]
    data_products = data[product_cols].fillna(0).astype(int)
    transactions = build_transactions(data_products)

    n = len(transactions)
    if n == 0:
        return pd.DataFrame(columns=["itemset", "support"]), {}, data_products

    tid_lists: dict[str, set[int]] = {}
    for tid, trans in enumerate(transactions):
        for item in trans:
            tid_lists.setdefault(item, set()).add(tid)

    support_cache: dict[frozenset[str], float] = {}
    frequent_itemsets: list[dict[str, object]] = []
    items = sorted(tid_lists.keys())

    for size in range(1, max_len + 1):
        for combo in combinations(items, size):
            tids = tid_lists[combo[0]].copy()
            for item in combo[1:]:
                tids &= tid_lists[item]
                if not tids:
                    break

            if not tids:
                continue

            support = len(tids) / n
            if support >= min_support:
                key = frozenset(combo)
                support_cache[key] = support
                frequent_itemsets.append({"itemset": key, "support": round(support, 4)})

    if not frequent_itemsets:
        return pd.DataFrame(columns=["itemset", "support"]), support_cache, data_products

    freq_df = (
        pd.DataFrame(frequent_itemsets)
        .sort_values(["support", "itemset"], ascending=[False, True])
        .reset_index(drop=True)
    )
    return freq_df, support_cache, data_products



### 5.2 Ejecución del modelo y consolidación de itemsets

Esta celda aplica ECLAT sobre la matriz transaccional final. El resultado principal es `freq_df`, que concentra los conjuntos frecuentes y su soporte observado, base cuantitativa para la fase de reglas de asociación.

In [56]:
freq_df, support_cache, productos_df = run_eclat(df_final)
print("Itemsets frecuentes:", len(freq_df))
display(freq_df.head(10))

Itemsets frecuentes: 339


,itemset,support
0,frozenset({carne}),0.740
1,frozenset({crema}),0.635
2,frozenset({arroz}),0.565
3,frozenset({frijol}),0.565
4,frozenset({pollo}),0.550
5,frozenset({pasta}),0.535
6,frozenset({harina}),0.530
7,frozenset({lenteja}),0.525
8,frozenset({leche}),0.510
9,frozenset({cafar}),0.505


## 6. Reglas de Asociación

**Fase equivalente en `scripts/eclat.ipynb`:** cálculo de `confidence`, `lift`, `affinity` y `score`.

Buenas prácticas aplicadas:
- lookup de soporte por diccionario (sin `.values[0]` frágil),
- reglas con formato legible para negocio,
- ranking dual por `affinity` y `score` para priorizar decisiones.

### 6.0 Construcción formal de reglas de asociación

La siguiente celda define la función generadora de reglas a partir de itemsets frecuentes. Se estiman soporte, confianza, lift, afinidad y score, incorporando filtros mínimos para mantener únicamente relaciones con significancia práctica.

In [ ]:
# Propósito: derivar reglas de asociación y métricas desde itemsets frecuentes.
def generate_rules(
    freq_itemsets_df: pd.DataFrame,
    support_map: dict[frozenset[str], float],
    min_confidence: float = 0.0,
    min_lift: float = 0.0,
) -> pd.DataFrame:
    """Genera reglas 1->N con soporte, confianza, lift, affinity y score."""
    rules: list[dict[str, object]] = []

    valid_rows = freq_itemsets_df[freq_itemsets_df["itemset"].apply(len) >= 2]
    for _, row in valid_rows.iterrows():
        itemset = row["itemset"]
        sup_itemset = support_map.get(itemset, float(row["support"]))

        for item in itemset:
            ant = frozenset([item])
            con = itemset - ant
            sup_ant = support_map.get(ant, 0.0)
            sup_con = support_map.get(con, 0.0)

            if sup_ant > 0 and sup_con > 0:
                confidence = sup_itemset / sup_ant
                lift = confidence / sup_con

                if confidence < min_confidence or lift < min_lift:
                    continue

                rules.append(
                    {
                        "antecedent": ", ".join(sorted(ant)),
                        "consequent": ", ".join(sorted(con)),
                        "support": round(sup_itemset, 4),
                        "confidence": round(confidence, 4),
                        "lift": round(lift, 4),
                        "affinity": round(confidence * lift, 4),
                        "score": round(
                            0.4 * sup_itemset + 0.3 * confidence + 0.3 * lift, 4
                        ),
                    }
                )

    if not rules:
        return pd.DataFrame(
            columns=[
                "antecedent",
                "consequent",
                "support",
                "confidence",
                "lift",
                "affinity",
                "score",
            ]
        )

    return (
        pd.DataFrame(rules)
        .sort_values(by=["affinity", "score"], ascending=[False, False])
        .reset_index(drop=True)
    )

### 6.1 Ejecución de reglas y selección de reglas relevantes

En esta celda se materializa la fase inferencial del modelo: se generan reglas desde itemsets frecuentes y se filtran por umbrales mínimos de **confianza** y **lift**. Esto garantiza que las reglas reportadas tengan relevancia estadística y potencial valor de negocio.

In [ ]:
# Propósito: ejecutar generación de reglas y exponer al menos 5 reglas relevantes con métricas clave.
rules_df = generate_rules(freq_df, support_cache, min_confidence=0.2, min_lift=1.0)

print('Reglas generadas:', len(rules_df))
print('Top 5 reglas relevantes (soporte, confianza y lift):')
display(rules_df[['antecedent', 'consequent', 'support', 'confidence', 'lift', 'affinity', 'score']].head(5))

print('Top 10 por affinity')
display(rules_df.sort_values('affinity', ascending=False).head(10))
print('Top 10 por score')
display(rules_df.sort_values('score', ascending=False).head(10))

## 7. Visualizaciones

**Fase equivalente en `scripts/eclat.ipynb`:** scatter, heatmap, histogramas y red de afinidad.

Objetivo de presentación: traducir patrones de co-compra en mensajes visuales comparables y accionables.

Secuencia analítica de las celdas:
1. **Dispersión soporte-confianza** para evaluar fuerza y cobertura de reglas.
2. **Mapa de calor de lift** para inspeccionar asociaciones de alta intensidad.
3. **Red de afinidad** para observar estructura relacional entre productos.
4. **Frecuencia de productos** para contextualizar popularidad base.
5. **Histogramas de métricas** para diagnóstico global del modelo.

In [ ]:
# Propósito: visualizar reglas con dispersión de soporte/confianza y tamaño por lift.
if not rules_df.empty:
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        rules_df["support"],
        rules_df["confidence"],
        s=rules_df["lift"] * 100,
        alpha=0.6,
        c=rules_df["lift"],
        cmap="RdYlGn",
    )
    plt.colorbar(scatter, label="Lift")
    plt.xlabel("Soporte")
    plt.ylabel("Confianza")
    plt.title("ECLAT — Reglas de Asociación (tamaño = Lift)")
    plt.show()
else:
    print("No hay reglas para visualizar en scatter.")

In [ ]:
# Propósito: generar heatmap de lift para top reglas por affinity.
if not rules_df.empty:
    top_heat = rules_df.head(15)
    heat_df = top_heat.pivot_table(index='antecedent', columns='consequent', values='lift', aggfunc='mean')

    plt.figure(figsize=(12, 8))
    sns.heatmap(heat_df, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
    plt.title('Heatmap de Lift (Top 15 reglas por affinity)')
    plt.tight_layout()
    plt.show()
else:
    print('No hay reglas para visualizar en heatmap.')

In [ ]:
# Propósito: dibujar red de afinidad entre productos con pesos por lift.
if not rules_df.empty:
    top_net = rules_df.head(20)
    graph = nx.DiGraph()

    for _, row in top_net.iterrows():
        ant = row['antecedent']
        con = row['consequent']
        lift = row['lift']
        graph.add_edge(ant, con, weight=lift)

    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(graph, seed=42)
    widths = [graph[u][v]['weight'] * 1.5 for u, v in graph.edges()]
    nx.draw_networkx_nodes(graph, pos, node_size=1400, node_color='lightblue')
    nx.draw_networkx_labels(graph, pos, font_size=9)
    nx.draw_networkx_edges(graph, pos, width=widths, edge_color='gray', arrows=True, alpha=0.7)
    plt.title('Red de afinidad (Top 20 reglas por affinity)')
    plt.axis('off')
    plt.show()
else:
    print('No hay reglas para visualizar en red de afinidad.')

In [ ]:
# Propósito: mostrar barras de frecuencia para top 20 productos.
product_freq = productos_df.sum().sort_values(ascending=False).head(20)
colors = sns.color_palette('viridis', n_colors=len(product_freq))

plt.figure(figsize=(10, 7))
plt.barh(product_freq.index[::-1], product_freq.values[::-1], color=colors)
plt.title('Top 20 productos por frecuencia')
plt.xlabel('Frecuencia')
plt.ylabel('Producto')
plt.tight_layout()
plt.show()

In [ ]:
# Propósito: mostrar distribución de métricas de reglas (diagnóstico de calidad del modelo).
if not rules_df.empty:
    rules_df[['support', 'confidence', 'lift']].hist(figsize=(12, 3), bins=20)
    plt.suptitle('Distribución de métricas en reglas ECLAT')
    plt.tight_layout()
    plt.show()
else:
    print('No hay reglas para distribución de métricas.')

## 8. Segmentación por Estrato NSE

**Fase equivalente en `scripts/eclat.ipynb`:** función `eclat_segmento` y comparación por estrato.

En esta versión se estandariza el cálculo de soporte y se separa la lógica en dos celdas independientes: primero cómputo por estrato y luego visualización comparativa.

In [ ]:
# Propósito: ejecutar ECLAT por estrato y construir comparativo de soporte por producto.
estrato_results: dict[str, pd.DataFrame] = {}
estrato_support_rows: list[pd.DataFrame] = []

for estrato in sorted(df_final['estrato_nse'].dropna().astype(str).unique()):
    subset = df_final[df_final['estrato_nse'].astype(str) == estrato]
    subset_freq, _, subset_products = run_eclat(subset, min_support=MIN_SUPPORT, max_len=MAX_ITEMSET)
    estrato_results[estrato] = subset_freq

    print(f'\nEstrato {estrato} | registros: {len(subset)}')
    display(subset_freq.head(5))

    product_support = (subset_products.sum() / max(len(subset_products), 1)).rename(estrato)
    estrato_support_rows.append(product_support.to_frame().T)

estrato_support_df = pd.concat(estrato_support_rows, axis=0).fillna(0)
common_top_10 = estrato_support_df.mean(axis=0).sort_values(ascending=False).head(10).index

### 8.1 Visualización comparativa por estrato

Esta celda desacopla la etapa de comunicación visual de la etapa de cómputo. Se construye un gráfico de barras agrupadas con los productos de mayor soporte promedio para facilitar comparaciones inter-estrato en contexto comercial.

In [ ]:
# Propósito: visualizar soporte de top productos por estrato NSE.
estrato_plot_df = estrato_support_df[common_top_10].T
estrato_plot_df.plot(kind='bar', figsize=(12, 6))
plt.title('Soporte por producto y estrato NSE (Top 10)')
plt.xlabel('Producto')
plt.ylabel('Soporte')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Segmentación por Grupo de Edad

**Fase equivalente en `scripts/eclat.ipynb`:** segmentación por `grupo_edad` con `pd.cut`.

Esta fase también se desacopla en celdas independientes: cálculo de soportes por cohorte etaria y visualización agregada para comparación entre ciclos de vida.

In [ ]:
# Propósito: ejecutar ECLAT por grupo etario y comparar soportes de productos.
edad_results: dict[str, pd.DataFrame] = {}
edad_support_rows: list[pd.DataFrame] = []

for grupo in df_final['grupo_edad'].dropna().astype(str).unique():
    subset = df_final[df_final['grupo_edad'].astype(str) == grupo]
    subset_freq, _, subset_products = run_eclat(subset, min_support=MIN_SUPPORT, max_len=MAX_ITEMSET)
    edad_results[grupo] = subset_freq

    print(f'\nGrupo {grupo} | registros: {len(subset)}')
    display(subset_freq.head(5))

    product_support = (subset_products.sum() / max(len(subset_products), 1)).rename(grupo)
    edad_support_rows.append(product_support.to_frame().T)

edad_support_df = pd.concat(edad_support_rows, axis=0).fillna(0)
common_age_top_10 = edad_support_df.mean(axis=0).sort_values(ascending=False).head(10).index

top_por_grupo = {
    grupo: soportes.sort_values(ascending=False).head(10).index.tolist()
    for grupo, soportes in edad_support_df.iterrows()
}
print('Top productos por grupo etario:')
display(pd.DataFrame.from_dict(top_por_grupo, orient='index').T)

### 9.1 Visualización comparativa por grupo de edad

Se representa el soporte de los productos con mayor frecuencia transversal por ciclo de vida. Este enfoque permite identificar patrones de consumo diferenciados entre cohortes etarias y facilita su comunicación en escenarios académicos y empresariales.

In [ ]:
# Propósito: visualizar soporte de top productos por grupo de edad.
edad_plot_df = edad_support_df[common_age_top_10].T
edad_plot_df.plot(kind='bar', figsize=(12, 6))
plt.title('Soporte por producto y grupo de edad (Top 10)')
plt.xlabel('Producto')
plt.ylabel('Soporte')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 10. Conclusiones y verificación académica

### Síntesis interpretativa

- El pipeline integra las fases observadas en `scripts/eclat.ipynb` (preparación, modelado, reglas, visualización y segmentación) en una narrativa única de presentación.
- La calidad de datos mejora con normalización lingüística, filtrado de ruido y separación estricta entre variables demográficas y productos.
- ECLAT se implementa con estructura vertical y `support_cache` O(1), evitando prácticas frágiles de búsqueda en DataFrame.
- La priorización combinada (`affinity` + `score`) permite traducir patrones técnicos en recomendaciones de negocio accionables.
- La segmentación por `estrato_nse` y `grupo_edad` evidencia variaciones de canasta útiles para surtido, promoción y comunicación segmentada.

### Verificación de los criterios solicitados

1. **Carga y descripción de datos**: cubierto en secciones de carga, fuente (`dataset.xlsx`) y caracterización de variables demográficas/producto.
2. **Preprocesamiento**: implementado con limpieza textual, normalización y transformación a formato transaccional one-hot.
3. **Modelo**: ECLAT aplicado con parámetros explícitos (`MIN_SUPPORT`, `MAX_ITEMSET`) y funciones reutilizables.
4. **Resultados**: se reportan reglas ordenadas con al menos 5 reglas relevantes y métricas de soporte, confianza y lift.
5. **Interpretación**: los hallazgos se explican en clave de decisiones reales y comparación por segmentos.
6. **Calidad del código**: celdas separadas por responsabilidad, comentarios de propósito y uso apropiado de librerías (`pandas`, `spacy`, `networkx`, `seaborn`).

### Recomendaciones para siguiente iteración

1. Validar sensibilidad de resultados para distintos umbrales (`MIN_SUPPORT`, `MAX_ITEMSET`).
2. Curar `replace_map` con hallazgos de baja frecuencia para mejorar cobertura semántica.
3. Versionar una tabla final de "Top reglas accionables" por segmento para comité comercial.